# 03 · Gold: P2P star schema

Builds the star schema the Direct Lake semantic model reads, plus a KPI snapshot.

| Table | Grain | Purpose |
|---|---|---|
| `gold_fact_po_item` | PO item | spend, requisition compliance, after-the-fact POs, delivery performance (OTIF), GR/IR |
| `gold_fact_supplier_invoice` | supplier invoice | first-time match, duplicate suspects, payment timeliness, open payables |
| `gold_dim_*` | one row per member | date (UK fiscal year), supplier, material, material group, plant, purchasing group |
| `gold_kpi_summary` | KPI | headline figures for the run, reconciled by notebook 04 |

KPI rules live in one cell below and are documented in `docs/kpi_definitions.md`.

In [ ]:
TABLE_FORMAT = "delta"
AS_OF_DATE = ""          # 'YYYY-MM-DD'; blank = extraction date of the latest bronze load
LOCAL_CURRENCY = "GBP"

In [ ]:
from datetime import date

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

OTIF_TOLERANCE_DAYS = 2          # delivered up to 2 days after the agreed date still counts as on time
IN_FULL_THRESHOLD = 0.98         # 98% of the ordered quantity counts as in full
DUPLICATE_WINDOW_DAYS = 90       # same supplier, reference and amount within 90 days = duplicate suspect
EARLY_PAYMENT_DAYS = 7           # paid more than 7 days before the due date = early (lost working capital)
GRIR_AGING_THRESHOLD_DAYS = 90

if not AS_OF_DATE:
    AS_OF_DATE = str(spark.table("bronze_ekko").agg(F.max("_extracted_as_of")).first()[0])
as_of = F.to_date(F.lit(AS_OF_DATE))
cent = F.lit(0.01).cast("decimal(18,2)")
in_full = F.lit(IN_FULL_THRESHOLD).cast("decimal(5,4)")
print(f"As-of date {AS_OF_DATE}, local currency {LOCAL_CURRENCY}")


def save(df: DataFrame, name: str):
    df.write.mode("overwrite").format(TABLE_FORMAT).option("overwriteSchema", "true").saveAsTable(name)
    print(f"{name:<32} {spark.table(name).count():>8,} rows")


fx = spark.table("silver_fx_rate").filter(F.col("to_currency") == LOCAL_CURRENCY) \
    .select("from_currency", "valid_from", "valid_to", "rate_to_local")


def add_local_amount(df: DataFrame, amount_col: str, currency_col: str, date_col: str, out_col: str) -> DataFrame:
    """Translate to local currency at the monthly rate valid on date_col."""
    joined = df.join(fx, (F.col(currency_col) == F.col("from_currency"))
                     & F.col(date_col).between(F.col("valid_from"), F.col("valid_to")), "left")
    rate = F.when(F.col(currency_col) == LOCAL_CURRENCY, F.lit(1)).otherwise(F.col("rate_to_local"))
    return (joined.withColumn(out_col, F.round(F.col(amount_col) * rate, 2).cast("decimal(18,2)"))
            .drop("from_currency", "valid_from", "valid_to", "rate_to_local"))

## Fact: PO item

In [ ]:
keys = ["purchase_order", "purchase_order_item"]
po = spark.table("silver_purchase_order").select(
    "purchase_order", "company_code", "supplier_id", "purchasing_group", "po_date", "currency", "payment_terms")
items = spark.table("silver_purchase_order_item").filter(~F.col("is_deleted"))
schedule = (spark.table("silver_purchase_order_schedule_line")
            .groupBy(keys).agg(F.min("delivery_date").alias("delivery_date")))
requisitions = spark.table("silver_purchase_requisition_item").select(
    "purchase_requisition", "purchase_requisition_item",
    F.coalesce("release_date", "requisition_date").alias("pr_release_date"))
history = spark.table("silver_purchase_order_history")

receipts = history.filter(F.col("event_type") == "1")
gr_totals = receipts.groupBy(keys).agg(
    F.sum("quantity").alias("gr_quantity"),
    F.sum("grir_clearing_value_local").alias("gr_value_gbp"),
    F.min(F.when(F.col("movement_type") == "101", F.col("posting_date"))).alias("first_gr_date"),
    F.max("posting_date").alias("last_gr_date"))
gr_in_window = (receipts.join(schedule, keys).groupBy(keys).agg(
    F.sum(F.when(F.col("posting_date") <= F.date_add("delivery_date", OTIF_TOLERANCE_DAYS), F.col("quantity"))
          .otherwise(F.lit(0))).alias("gr_quantity_in_window")))
ir_totals = history.filter(F.col("event_type") == "2").groupBy(keys).agg(
    F.sum("quantity").alias("ir_quantity"),
    F.sum("grir_clearing_value_local").alias("ir_value_gbp"))
first_invoice = (spark.table("silver_supplier_invoice_item")
                 .join(spark.table("silver_supplier_invoice").select("invoice_key", "document_date"), "invoice_key")
                 .groupBy(keys).agg(F.min("document_date").alias("first_invoice_document_date")))

fact = (items
        .join(po, "purchase_order")
        .join(schedule, keys, "left")
        .join(requisitions, ["purchase_requisition", "purchase_requisition_item"], "left")
        .join(gr_totals, keys, "left")
        .join(gr_in_window, keys, "left")
        .join(ir_totals, keys, "left")
        .join(first_invoice, keys, "left"))
fact = add_local_amount(fact, "net_amount", "currency", "po_date", "net_amount_gbp")

on_time_cutoff = F.date_add("delivery_date", OTIF_TOLERANCE_DAYS)
grir_balance = F.coalesce("gr_value_gbp", F.lit(0)) - F.coalesce("ir_value_gbp", F.lit(0))
fact = (fact
        .withColumn("po_item_key", F.concat("purchase_order", "purchase_order_item"))
        .withColumn("material_id", F.when(F.col("material_id") == "", "#").otherwise(F.col("material_id")))
        .withColumn("is_pr_backed", F.col("purchase_requisition") != "")
        .withColumn("pr_to_po_days", F.when(F.col("is_pr_backed"), F.datediff("po_date", "pr_release_date")))
        .withColumn("is_after_the_fact", F.coalesce(F.col("first_invoice_document_date") < F.col("po_date"), F.lit(False)))
        .withColumn("is_due", F.coalesce(on_time_cutoff <= as_of, F.lit(False)))
        .withColumn("is_on_time", F.col("is_due") & F.coalesce(F.col("first_gr_date") <= on_time_cutoff, F.lit(False)))
        .withColumn("is_otif", F.col("is_due") & (F.coalesce("gr_quantity_in_window", F.lit(0))
                                                 >= F.col("order_quantity") * in_full))
        .withColumn("po_to_first_gr_days", F.datediff("first_gr_date", "po_date"))
        .withColumn("grir_balance_gbp", grir_balance.cast("decimal(18,2)"))
        .withColumn("grir_status", F.when(F.col("grir_balance_gbp") >= cent, "GR not invoiced")
                                    .when(F.col("grir_balance_gbp") <= -cent, "Invoiced not received")
                                    .otherwise("Balanced"))
        .withColumn("grir_age_days", F.when(F.col("grir_status") == "GR not invoiced", F.datediff(as_of, "last_gr_date")))
        .withColumn("grir_age_bucket", F.when(F.col("grir_age_days").isNull(), F.lit(None))
                                        .when(F.col("grir_age_days") <= 30, "0-30")
                                        .when(F.col("grir_age_days") <= 60, "31-60")
                                        .when(F.col("grir_age_days") <= GRIR_AGING_THRESHOLD_DAYS, "61-90")
                                        .otherwise("90+")))

fact_po_item = fact.select(
    "po_item_key", "purchase_order", "purchase_order_item", "company_code", "supplier_id", "purchasing_group",
    "plant", "material_id", "material_group", "account_assignment_category", "item_text",
    "po_date", "delivery_date", "currency", "payment_terms", "order_quantity", "order_unit", "net_unit_price",
    "net_amount", "net_amount_gbp",
    "is_pr_backed", "purchase_requisition", "pr_release_date", "pr_to_po_days",
    "first_invoice_document_date", "is_after_the_fact",
    "first_gr_date", "last_gr_date", "po_to_first_gr_days", "gr_quantity", "gr_quantity_in_window",
    "is_due", "is_on_time", "is_otif", "is_delivery_completed",
    "ir_quantity", "gr_value_gbp", "ir_value_gbp", "grir_balance_gbp", "grir_status", "grir_age_days",
    "grir_age_bucket")
save(fact_po_item, "gold_fact_po_item")

## Fact: supplier invoice

In [ ]:
payments = (spark.table("silver_supplier_invoice_payment")
            .filter(F.col("clearing_date") <= as_of)
            .groupBy("invoice_key").agg(F.max("clearing_date").alias("payment_date")))
same_invoice = (Window.partitionBy("supplier_id", "supplier_reference_normalised", "gross_amount")
                .orderBy("posting_date", "fiscal_year", "supplier_invoice"))

invoices = add_local_amount(spark.table("silver_supplier_invoice"), "gross_amount", "currency", "posting_date",
                            "gross_amount_gbp")
fact_invoice = (invoices
                .join(payments, "invoice_key", "left")
                .withColumn("_previous_posting", F.lag("posting_date").over(same_invoice))
                .withColumn("_previous_invoice", F.lag("invoice_key").over(same_invoice))
                .withColumn("is_duplicate_suspect", F.coalesce(
                    F.datediff("posting_date", "_previous_posting") <= DUPLICATE_WINDOW_DAYS, F.lit(False)))
                .withColumn("duplicate_of_invoice", F.when(F.col("is_duplicate_suspect"), F.col("_previous_invoice")))
                .withColumn("is_paid", F.col("payment_date").isNotNull())
                .withColumn("days_to_pay", F.when(F.col("is_paid"), F.datediff("payment_date", "document_date")))
                .withColumn("is_paid_on_time", F.col("is_paid") & (F.col("payment_date") <= F.col("due_date")))
                .withColumn("is_paid_early", F.col("is_paid")
                            & (F.col("payment_date") < F.date_sub("due_date", EARLY_PAYMENT_DAYS)))
                .withColumn("is_open", ~F.col("is_paid"))
                .withColumn("is_overdue", F.col("is_open") & (F.col("due_date") < as_of))
                .withColumn("days_overdue", F.when(F.col("is_overdue"), F.datediff(as_of, "due_date")))
                .select(
                    "invoice_key", "supplier_invoice", "fiscal_year", "company_code", "supplier_id",
                    "supplier_reference", "supplier_reference_normalised", "document_date", "posting_date",
                    "currency", "net_amount", "tax_amount", "gross_amount", "gross_amount_gbp",
                    "payment_terms", "baseline_date", "due_date", "is_payment_blocked", "payment_block_reason",
                    "is_duplicate_suspect", "duplicate_of_invoice",
                    "payment_date", "is_paid", "days_to_pay", "is_paid_on_time", "is_paid_early",
                    "is_open", "is_overdue", "days_overdue"))
save(fact_invoice, "gold_fact_supplier_invoice")

## Dimensions

In [ ]:
first_day = spark.table("gold_fact_po_item").agg(F.min("po_date")).first()[0]
first_invoice_day = spark.table("gold_fact_supplier_invoice").agg(F.min("document_date")).first()[0]
start = date(min(first_day, first_invoice_day).year, 1, 1)
end = date(date.fromisoformat(AS_OF_DATE).year, 12, 31)

fiscal_start_year = F.when(F.month("date") >= 4, F.year("date")).otherwise(F.year("date") - 1)
dim_date = (spark.sql(f"SELECT explode(sequence(DATE'{start}', DATE'{end}', INTERVAL 1 DAY)) AS date")
            .select(
                "date",
                F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
                F.year("date").alias("year"),
                F.concat(F.lit("Q"), F.quarter("date")).alias("quarter"),
                F.month("date").alias("month"),
                F.date_format("date", "MMM").alias("month_name"),
                F.date_format("date", "yyyy-MM").alias("year_month"),
                F.trunc("date", "month").alias("month_start"),
                F.weekofyear("date").alias("iso_week"),
                ((F.dayofweek("date") + 5) % 7 + 1).alias("day_of_week"),   # Monday = 1
                F.date_format("date", "EEE").alias("day_name"),
                (F.dayofweek("date").isin(1, 7)).alias("is_weekend"),
                # UK fiscal year April-March, e.g. FY2025/26
                F.concat(F.lit("FY"), fiscal_start_year.cast("string"),
                         F.lit("/"), F.lpad(((fiscal_start_year + 1) % 100).cast("string"), 2, "0")).alias("fiscal_year"),
                ((F.month("date") + 8) % 12 + 1).alias("fiscal_period")))
save(dim_date, "gold_dim_date")

supplier_unknown = spark.createDataFrame(
    [("#", "#", "Unknown supplier", "", "", False)],
    "supplier_id string, supplier_number string, supplier_name string, country_code string, city string, "
    "is_purchasing_blocked boolean")
save(spark.table("silver_supplier")
     .select("supplier_id", "supplier_number", "supplier_name", "country_code", "city", "is_purchasing_blocked")
     .unionByName(supplier_unknown), "gold_dim_supplier")

material_unknown = spark.createDataFrame(
    [("#", "#", "Service / non-stock item", "", "", "")],
    "material_id string, material_number string, material_description string, material_type string, "
    "material_group string, base_unit string")
save(spark.table("silver_material")
     .select("material_id", "material_number", "material_description", "material_type", "material_group", "base_unit")
     .unionByName(material_unknown), "gold_dim_material")

# Direct spend = goes into the product (ingredients, packaging); in production this mapping is a maintained table.
save(spark.table("silver_material_group")
     .withColumn("spend_type", F.when(F.col("material_group").rlike("^(ING|PKG)-"), "Direct").otherwise("Indirect")),
     "gold_dim_material_group")
save(spark.table("silver_plant"), "gold_dim_plant")
save(spark.table("silver_purchasing_group"), "gold_dim_purchasing_group")

## KPI snapshot
The same figures the report shows, computed once per run so notebook 04 can reconcile them.

In [ ]:
p = spark.table("gold_fact_po_item")
i = spark.table("gold_fact_supplier_invoice")
grir_open = F.col("grir_status") == "GR not invoiced"


def total(condition, value=None):
    return F.coalesce(F.sum(F.when(condition, value if value is not None else F.lit(1))), F.lit(0))


po_kpis = p.agg(
    F.count("*").alias("po_items"),
    F.sum("net_amount_gbp").alias("spend_gbp"),
    F.avg(F.col("is_pr_backed").cast("int")).alias("pr_backed_rate"),
    F.avg("pr_to_po_days").alias("avg_pr_to_po_days"),
    total(F.col("is_after_the_fact")).alias("after_the_fact_items"),
    total(F.col("is_due")).alias("due_items"),
    total(F.col("is_on_time")).alias("on_time_items"),
    total(F.col("is_otif")).alias("otif_items"),
    F.avg("po_to_first_gr_days").alias("avg_po_to_first_gr_days"),
    total(grir_open).alias("grir_open_items"),
    total(grir_open, F.col("grir_balance_gbp")).alias("grir_gr_not_invoiced_gbp"),
    total(F.col("grir_status") == "Invoiced not received", -F.col("grir_balance_gbp")).alias("grir_invoiced_not_received_gbp"),
    total(F.col("grir_age_bucket") == "90+", F.col("grir_balance_gbp")).alias("grir_over_90_days_gbp"),
).first().asDict()

invoice_kpis = i.agg(
    F.count("*").alias("invoices"),
    F.avg(F.col("is_payment_blocked").cast("int")).alias("blocked_invoice_rate"),
    total(F.col("is_duplicate_suspect")).alias("duplicate_invoice_suspects"),
    total(F.col("is_paid")).alias("paid_invoices"),
    total(F.col("is_paid_on_time")).alias("paid_on_time_invoices"),
    total(F.col("is_paid_early")).alias("paid_early_invoices"),
    F.avg("days_to_pay").alias("avg_days_to_pay"),
    total(F.col("is_open")).alias("open_invoices"),
    total(F.col("is_open"), F.col("gross_amount_gbp")).alias("open_invoice_value_gbp"),
    total(F.col("is_overdue")).alias("overdue_open_invoices"),
).first().asDict()

supplier_spend = p.groupBy("supplier_id").agg(F.sum("net_amount_gbp").alias("spend"))
top10 = sum(r["spend"] for r in supplier_spend.orderBy(F.desc("spend")).limit(10).collect())

kpis = {
    "po_items": po_kpis["po_items"],
    "spend_gbp": po_kpis["spend_gbp"],
    "pr_backed_rate": po_kpis["pr_backed_rate"],
    "avg_pr_to_po_days": po_kpis["avg_pr_to_po_days"],
    "after_the_fact_items": po_kpis["after_the_fact_items"],
    "due_items": po_kpis["due_items"],
    "on_time_rate": po_kpis["on_time_items"] / po_kpis["due_items"],
    "otif_rate": po_kpis["otif_items"] / po_kpis["due_items"],
    "avg_po_to_first_gr_days": po_kpis["avg_po_to_first_gr_days"],
    "grir_open_items": po_kpis["grir_open_items"],
    "grir_gr_not_invoiced_gbp": po_kpis["grir_gr_not_invoiced_gbp"],
    "grir_invoiced_not_received_gbp": po_kpis["grir_invoiced_not_received_gbp"],
    "grir_over_90_days_gbp": po_kpis["grir_over_90_days_gbp"],
    "invoices": invoice_kpis["invoices"],
    "blocked_invoice_rate": invoice_kpis["blocked_invoice_rate"],
    "duplicate_invoice_suspects": invoice_kpis["duplicate_invoice_suspects"],
    "paid_invoices": invoice_kpis["paid_invoices"],
    "paid_on_time_rate": invoice_kpis["paid_on_time_invoices"] / invoice_kpis["paid_invoices"],
    "paid_early_rate": invoice_kpis["paid_early_invoices"] / invoice_kpis["paid_invoices"],
    "avg_days_to_pay": invoice_kpis["avg_days_to_pay"],
    "open_invoices": invoice_kpis["open_invoices"],
    "open_invoice_value_gbp": invoice_kpis["open_invoice_value_gbp"],
    "overdue_open_invoices": invoice_kpis["overdue_open_invoices"],
    "top10_supplier_spend_share": top10 / po_kpis["spend_gbp"],
}
summary = spark.createDataFrame([(k, float(v)) for k, v in kpis.items()], "metric string, value double") \
    .withColumn("as_of_date", as_of).withColumn("computed_at", F.current_timestamp())
save(summary, "gold_kpi_summary")
for k, v in kpis.items():
    print(f"  {k:<34} {float(v):>18,.4f}")